# Backtest — Reversion a la Media Donchian sobre Binance USD-M Perpetuals (Piloto Coinbase50)

Validacion independiente del port de la estrategia de reversion a la media (fade de ruptura Donchian,
sizing/piramidado estilo Turtle) desde CME Bitcoin futures (IBKR) hacia Binance USDⓈ-M perpetuals,
sobre el universo piloto de 5 monedas (`qsCrypto/config/universe_coinbase50_binance.yaml`):
BTC, ETH, SOL, AVAX, LINK.

Sigue el flujo de trabajo del skill `backtester`: declarar sesgos -> datos reales -> motor con costes
reales -> baseline sin optimizar -> walk-forward con purga/embargo -> cuantificacion de sesgos ->
Deflated Sharpe / PBO -> checklist -> veredicto.

**Esto es una validacion INDEPENDIENTE, no una copia del resultado de
`MeanReversion_IBKR_CME_Bitcoin.ipynb`.** Esa validacion en CME ya reporta fallar 4/12 criterios de
viabilidad (ver `Turtle_MR_Intraday_Live.ipynb`). El universo, la microestructura (24/7 sin sesiones),
el modelo de costes (fees en bps + funding recurrente en vez de comision fija en USD/contrato +
slippage en ticks) y la granularidad de ruido de un perpetuo cripto son genuinamente distintos: no se
puede asumir viabilidad por analogia.

## Tabla de sesgos declarados

| # | Sesgo | Impacto | Control en este backtest |
|---|-------|---------|---------------------------|
| 1 | Survivorship bias | 🟡 Moderado | Universo piloto = 5 monedas listadas en Binance perps desde antes de 2022-01-01, todas "veteranas" en la muestra completa. El sesgo residual esta en la CURACION del piloto (elegidas por liquidez/market-cap ACTUAL), no en la construccion de la muestra — ver §6.5. |
| 2 | Look-ahead | 🟢 Controlado | `compute_n`/canal Donchian se reutilizan LITERALMENTE del modulo de estrategia en vivo (`qsCrypto.strategy.strategy`), no se re-derivan. Test de truncamiento ejecutado (§2) y test de PARIDAD bit-a-bit contra `MeanReversionFadeStrategy` en vivo (ver commit `5e1edfc`) — ambos en verde. Se detecto y corrigio un bug real de look-ahead de una barra durante el desarrollo del motor (el fill se ejecutaba una barra antes de lo que el sistema en vivo dispararia la orden). |
| 3 | Overfitting | 🔴 Severo (medido) | Rejilla deliberadamente pequeña de 6 configuraciones (`entry_bars` x `stop_n`); DSR y PBO calculados con `n_trials=6` declarado — ver §5. |
| 4 | Costes ignorados | 🟢 Controlado | Fees en bps de nocional (taker) + funding REAL historico (`GET /fapi/v1/fundingRate`) aplicado en cada liquidacion de 8h — nunca un coste fijo en USD como en el backtest de CME. Barrido de sensibilidad en §6.3. |

**El diagnostico que importa aqui NO es el Sharpe — es que la estrategia pierde dinero incluso en el
escenario "Sin costes" (§6.3): el problema es de EDGE, no de friccion.**

## 1. Datos — reales, cacheados en disco

`GET /fapi/v1/klines` (30m) y `GET /fapi/v1/fundingRate`, ambos endpoints publicos de Binance
(sin API key), paginados y cacheados en `data_cache/binance/{30m,funding}/` — mismo patron que
`data_cache/ibkr/{barSize}/`. ~4.7 años de historia real por simbolo, 2022-01-01 hasta hoy.

In [1]:
from qsCrypto.backtest.run_pilot_validation import main
results = main()


1. Data
  BTCUSDT: 82212 bars, 2022-01-01 00:00:00 -> 2026-09-09 17:30:00 | 5139 funding settlements
  ETHUSDT: 82212 bars, 2022-01-01 00:00:00 -> 2026-09-09 17:30:00 | 5139 funding settlements
  SOLUSDT: 82212 bars, 2022-01-01 00:00:00 -> 2026-09-09 17:30:00 | 5214 funding settlements
  AVAXUSDT: 82212 bars, 2022-01-01 00:00:00 -> 2026-09-09 17:30:00 | 5139 funding settlements
  LINKUSDT: 82212 bars, 2022-01-01 00:00:00 -> 2026-09-09 17:30:00 | 5139 funding settlements

2. Anti-look-ahead truncation test (compute_n + Donchian channel)
  compute_n: 30/30 truncation checks pass
  channel  : 30/30 truncation checks pass

3. Baseline (un-optimized canonical parameters), per symbol
  BTCUSDT: CAGR=-100.00% Sharpe=-6.38 MaxDD=-100.00% Trades=4769 WinRate=18.35%
  ETHUSDT: CAGR=-100.00% Sharpe=-5.09 MaxDD=-100.00% Trades=4605 WinRate=24.56%
  SOLUSDT: CAGR=-100.00% Sharpe=-3.65 MaxDD=-100.00% Trades=4907 WinRate=27.21%
  AVAXUSDT: CAGR=-100.00% Sharpe=-3.42 MaxDD=-100.00% Trades=4933 WinRat

## Resumen de resultados (numeros reales de la celda anterior)

### Baseline por simbolo (parametros canonicos Turtle sin optimizar)

| Simbolo | CAGR | Sharpe | Max DD | Trades | Win rate |
|---|---|---|---|---|---|
| BTCUSDT | -100.0% | -6.38 | -100.0% | 4769 | 18.3% |
| ETHUSDT | -100.0% | -5.09 | -100.0% | 4605 | 24.6% |
| SOLUSDT | -100.0% | -3.65 | -100.0% | 4907 | 27.2% |
| AVAXUSDT | -100.0% | -3.42 | -100.0% | 4933 | 26.7% |
| LINKUSDT | -100.0% | -2.56 | -100.0% | 5121 | 26.3% |

Las cinco monedas, con los MISMOS parametros que se validaron en CME (canal Donchian de 20 barras,
N de 20 periodos, stop 1N, objetivo en el centro del canal, piramidado Turtle hasta 4 unidades),
pierden dinero de forma sostenida a 30 minutos en Binance perpetuals — no es un problema de una
sola moneda.

### Walk-forward (14 folds, IS=365d/OOS=90d/embargo=2d, sobre BTCUSDT)

- Config ganadora por seleccion IS penalizada (Sharpe_IS - 0.5·|MaxDD_IS|): estable en
  **{'entry_bars': 15, 'stop_n': 1.0}**, elegida en
  13/14 folds — la optimizacion SI converge a un parametro estable, pero converge al "menos malo", no a uno rentable.
- t-test OOS (seleccionada vs. baseline fija, misma ventana): p=0.0003,
  win_rate=7.1% — la diferencia es significativa, pero en la
  direccion equivocada: la config "ganadora" gana MENOS negativo, no positivo (aviso de muestra:
  n<20 folds: diagnóstico cualitativo, no evidencia).

### Deflated Sharpe Ratio y PBO (n_trials=6, declarado)

- DSR: sharpe=-2.66, benchmark=0.40,
  **dsr=0.000 -> NO distinguible de ruido**.
- PBO (CSCV, 10 particiones): **pbo=0.000 -> selección con valor**
  — la seleccion IS→OOS es consistente (no es azar), lo cual es coherente con la estabilidad de
  parametros de arriba: el proceso de seleccion funciona, simplemente no hay ningun punto de la
  rejilla que sea rentable.

### Cuantificacion de sesgos

- **Concentracion de P&L**: top_3=-5.3% del P&L total (con signo negativo, ya que el
  resultado neto es una perdida) — concentración aceptable. La perdida esta distribuida
  en miles de trades, no concentrada en unos pocos — descarta "mala suerte puntual" como explicacion.
- **Apuestas efectivas** (5 monedas del piloto): crees tener 5 mercados; tienes ~2
  (correlacion media=0.75). Crees tener 5 mercados
  descorrelacionados; en la practica el piloto se comporta como ~1.5 apuestas independientes —
  BTC/ETH/SOL/AVAX/LINK se mueven juntos en el regimen de riesgo cripto, y un piloto "multi-moneda"
  no diversifica tanto como su nombre sugiere.
- **Sensibilidad a costes**: pierde dinero INCLUSO en el escenario "Sin costes" (CAGR=-58.6% en 4.7
  años) — el edge no muere por friccion, nace negativo. Con costes base (5bps fee + 10bps slippage +
  funding real) el resultado se agrava a ruina total. False
- **Monte Carlo** (bootstrap con reemplazo de los trades de BTCUSDT, 5000 sims): probabilidad de
  ruina (DD>=50%) = 100% — no es una cola de riesgo, es el
  resultado central esperado.
- **Survivorship**: All pilot coins pre-date the sample window; residual survivorship risk is in curating the pilot list by current liquidity, not in-sample listing bias.

## 7. Checklist de validez — veredicto

- [ ] wfo_beats_baseline_60pct
- [x] wfo_diff_significant_p05
- [x] params_stable_gt50pct
- [ ] dsr_gt_0_at_95
- [x] pbo_lt_0.5
- [ ] survives_pessimistic_costs
- [x] top3_trades_lt_50pct
- [ ] effective_bets_gte_8
- [ ] mc_ruin_prob_lt_20pct
- [x] lookahead_tests_pass

**5/10 criterios superados.**

## VEREDICTO: NO DESPLIEGUES CAPITAL

### Diagnostico estructural (no es un problema de parametros)

1. **El edge nace negativo en Binance a 30 minutos, no lo mata el coste.** El escenario "Sin costes"
   ya pierde -58.6% de CAGR sobre 4.7 años en BTCUSDT. Ajustar fees o slippage no puede arreglar esto:
   el problema es la señal, no la friccion.
2. **El sizing de riesgo fijo (fixed-fractional) compuesto sobre miles de trades con edge ligeramente
   negativo es letal.** Con ~4700 trades por simbolo y un r-multiple medio de aproximadamente -0.27R,
   el sizing de Kelly-fraccional (arriesgar 1% del EQUITY ACTUAL, no del capital inicial) hace que la
   perdida componga multiplicativamente hacia cero — un mismo edge ligeramente negativo bajo sizing
   fijo en DOLARES habria perdido dinero mas lento, pero fijo-fraccional amplifica exactamente lo que
   el skill `turtlerisk`/`MR_BTC_System` advierte que hace bien cuando el edge es positivo: componer
   rapido. Compone igual de rapido cuando es negativo.
3. **Los parametros (canal de 20 barras, N de 20 periodos, confirmacion de 1 barra) se calibraron y
   validaron contra el ruido de 30 minutos de CME MBT — un mercado con sesiones, sin funding, y con
   una microestructura de futuros regulados.** Binance USDⓈ-M a 30 minutos es un mercado 24/7, sin
   pausas de sesion que "limpien" el ruido intradia, con mas actividad algoritmica de alta frecuencia
   compitiendo por el mismo canal de ruptura. La frecuencia de señales observada aqui (~1 cada 17
   barras, frente a mucho menor en el backtest de CME) es sintoma de que el canal de 20 barras se
   rompe con mucha mas frecuencia en cripto, generando muchas mas señales de calidad marginal.
4. **Esto no lo arregla mas optimizacion.** La rejilla de 6 configuraciones SI converge de forma
   estable (PBO=0, moda de parametro ganador en 13/14 folds) — el proceso de seleccion funciona
   correctamente. El problema es que NINGUN punto de la rejilla probada es rentable: no hay una
   variante de este mismo enfoque, con estos parametros calibrados para CME, que sobreviva el
   transplante a Binance perpetuals a 30 minutos sin una revision de fondo de la logica de señal o
   la granularidad.

### Lo que NO se recomienda hacer a continuacion

- No reoptimizar `entry_bars`/`stop_n` sobre esta misma muestra: el checklist ya muestra que la
  optimizacion converge de forma estable a un parametro que sigue siendo negativo (PBO bajo no es lo
  mismo que rentable).
- No proceder a testnet/vivo con esta configuracion — el go/no-go del plan de migracion exige un
  veredicto de backtest positivo ANTES del paso de testnet, y este es explicitamente negativo.

### Lineas de investigacion honestas para retomar esto (fuera del alcance de esta validacion)

- Recalibrar `entry_bars`/`atr_period` especificamente para la microestructura de 30 minutos de
  Binance (no asumir que los valores de CME transfieren), con una MUESTRA DE VALIDACION NUEVA e
  independiente de la usada aqui para evitar que la propia recalibracion sea la siguiente prueba de
  overfitting no declarada.
- Probar una granularidad distinta (1h, 4h) donde el ruido de microestructura de alta frecuencia pese
  menos frente a la señal de reversion.
- Revisar si el modo de piramidado Turtle (que compone riesgo hacia el estilo tendencial) es
  estructuralmente inadecuado para una estrategia de FADE — el diseño Turtle original piramida
  siguiendo tendencia, no en contra de ella.